# Bayesian Optimisation (BO)

Bayesian optimisation is a very sample-efficient way of maximising parameters. In this example, the yield of a chemical reaction in dependence of two features, temperature and catalyst concentration, shall be maximised.

To do so, the real function *yield(temperature, cat. concentration)* is sampled by conducting a few experiments to start with. The BO will then suggest the next conditions to try experimentally. This loop is repeated until the maximum is closely enough approximated.

To mimic this process here in this notebook, the yield function is hardcoded and randomly sampled. Note that this function is for demonstration purposes and does of course not correspond to real reaction behaviour.

### 0) import dependencies

In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from scipy.stats import norm

### 1) Define the "hidden" yield function

The mean yield is simply defined as a product of two parametrised exponential functions. Some random noise is created as well to make the data a bit more lifelike.

In [2]:
def reaction_yield(x):
    """
    temp = temperature feature
    cat = catalyst concentration feature
    """
    temp, cat = x
    yield_mean = (
        np.exp(-0.002*(temp - 85)**2) *
        np.exp(-5*(cat - 0.8)**2)
    )
    noise = np.random.normal(0, 0.02) # random samples from a normal distribution (mean, std)
    return yield_mean + noise

### 2) Initiate experimental design

In [3]:
# define process boundaries
bounds = np.array([
    [40, 120],    # temperature
    [0.1, 2.0]    # catalyst concentration
])

def sample_random(n=5): # default value defined - if not stated explicetly, sample 5 points
    return np.random.uniform(bounds[:,0], bounds[:,1], size=(n, 2))


X = sample_random(6)
y = np.array([reaction_yield(x) for x in X])

### 3) Fit Gaussian process

In [4]:
kernel = RBF(length_scale=[10, 0.3]) + WhiteKernel(noise_level=0.01)

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True)
gp.fit(X, y)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",RBF(length_sc...se_level=0.01)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",0
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",True
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`Gl

### 4) Calculate expected improvement

In [ ]:
def expected_improvement(X_candidate, model, y_best, xi=0.01): # xi...exploration parameter
    mu, sigma = model.predict(X_candidate, return_std=True)
    sigma = sigma.reshape(-1, 1)

    improvement = mu.reshape(-1,1) - y_best - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei.ravel()

### 5) Optimisation loop

In [6]:
# create a grid for the predicted conditions
temp_grid = np.linspace(40, 120, 200)
cat_grid = np.linspace(0.1, 2.0, 200)

T, C = np.meshgrid(temp_grid, cat_grid)
X_grid = np.column_stack([T.ravel(), C.ravel()])

In [7]:
n_iterations = 15

for i in range(n_iterations):
    # Acquisition step
    X_cand = sample_random(500)
    ei = expected_improvement(X_cand, gp, y.max())
    x_next = X_cand[np.argmax(ei)]

    # Run experiment 
    y_next = reaction_yield(x_next)

    # Update data 
    X = np.vstack([X, x_next])
    y = np.append(y, y_next)

    # Refit GP
    gp.fit(X, y)

    # GP-predicted optimum 
    mu, _ = gp.predict(X_grid, return_std=True)
    idx_gp = np.argmax(mu)
    temp_gp, cat_gp = X_grid[idx_gp]

    # Best observed so far
    idx_obs = np.argmax(y)
    temp_obs, cat_obs = X[idx_obs]

    # Printout
    print(f" Iteration {i+1}")
    print(f"  Best observed yield: {y[idx_obs]:.3f} (Temp = {temp_obs:.1f} °C, Cat = {cat_obs:.3f} mol%)")
    print(f"  GP-predicted optimum: (Temp = {temp_gp:.1f} °C, Cat = {cat_gp:.3f} mol%)")

 Iteration 1
  Best observed yield: 0.203 (Temp = 112.2 °C, Cat = 0.985 mol%)
  GP-predicted optimum: (Temp = 118.8 °C, Cat = 1.017 mol%)
 Iteration 2
  Best observed yield: 0.203 (Temp = 112.2 °C, Cat = 0.985 mol%)
  GP-predicted optimum: (Temp = 112.4 °C, Cat = 0.978 mol%)
 Iteration 3
  Best observed yield: 0.203 (Temp = 112.2 °C, Cat = 0.985 mol%)
  GP-predicted optimum: (Temp = 118.8 °C, Cat = 0.100 mol%)
 Iteration 4
  Best observed yield: 0.203 (Temp = 112.2 °C, Cat = 0.985 mol%)
  GP-predicted optimum: (Temp = 112.4 °C, Cat = 0.635 mol%)
 Iteration 5
  Best observed yield: 0.203 (Temp = 112.2 °C, Cat = 0.985 mol%)
  GP-predicted optimum: (Temp = 112.4 °C, Cat = 0.635 mol%)


c:\Users\jschoer\Desktop\DSA104development\DSA104\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


 Iteration 6
  Best observed yield: 0.205 (Temp = 112.4 °C, Cat = 0.655 mol%)
  GP-predicted optimum: (Temp = 112.4 °C, Cat = 0.816 mol%)
 Iteration 7
  Best observed yield: 0.212 (Temp = 113.2 °C, Cat = 0.812 mol%)
  GP-predicted optimum: (Temp = 110.8 °C, Cat = 0.845 mol%)
 Iteration 8
  Best observed yield: 0.348 (Temp = 107.0 °C, Cat = 0.815 mol%)
  GP-predicted optimum: (Temp = 103.9 °C, Cat = 0.826 mol%)
 Iteration 9
  Best observed yield: 0.633 (Temp = 99.6 °C, Cat = 0.841 mol%)
  GP-predicted optimum: (Temp = 97.1 °C, Cat = 0.768 mol%)
 Iteration 10
  Best observed yield: 0.710 (Temp = 96.5 °C, Cat = 0.705 mol%)
  GP-predicted optimum: (Temp = 90.7 °C, Cat = 0.759 mol%)
 Iteration 11
  Best observed yield: 0.977 (Temp = 87.4 °C, Cat = 0.783 mol%)
  GP-predicted optimum: (Temp = 83.8 °C, Cat = 0.873 mol%)
 Iteration 12
  Best observed yield: 0.977 (Temp = 87.4 °C, Cat = 0.783 mol%)
  GP-predicted optimum: (Temp = 85.4 °C, Cat = 0.778 mol%)
 Iteration 13
  Best observed yield: 0.

### Discussion points

- How does BO compare to grid or random search?
- What happens if noise increases?
- How does more exploration influence the model's behaviour?
- How would you add constraints (e.g. toxicity)?
- Estimate many experiments is BO saving as compared to grid search?